<a href="https://colab.research.google.com/github/salia40/mini-project-salia/blob/main/mini%20projeck%20web%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
import subprocess
import time
import re

# 1. Install dependensi
print("--- [1/4] Menginstal PHP, MySQL, & Cloudflare Tunnel ---")
os.system("apt-get update -qq")
os.system("apt-get install mysql-server php php-mysql wget -y -qq")
os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
os.system("chmod +x cloudflared")

# 2. Setup Database & Password Hash yang Valid
print("--- [2/4] Menyiapkan Database MySQL & Tabel ---")
os.system("service mysql start")

admin_hash = subprocess.check_output(["php", "-r", "echo password_hash('admin123', PASSWORD_DEFAULT);"]).decode().strip()
staff_hash = subprocess.check_output(["php", "-r", "echo password_hash('staff123', PASSWORD_DEFAULT);"]).decode().strip()

schema_sql = f"""
DROP DATABASE IF EXISTS aksesoris_inventory;
CREATE DATABASE aksesoris_inventory;
USE aksesoris_inventory;

CREATE TABLE users (
    id INT AUTO_INCREMENT PRIMARY KEY,
    username VARCHAR(50) NOT NULL UNIQUE,
    password_hash VARCHAR(255) NOT NULL,
    role ENUM('admin', 'staff') NOT NULL
);

CREATE TABLE categories (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100) NOT NULL
);

CREATE TABLE suppliers (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    phone VARCHAR(20)
);

CREATE TABLE products (
    id INT AUTO_INCREMENT PRIMARY KEY,
    category_id INT NOT NULL,
    sku VARCHAR(50) NOT NULL UNIQUE,
    name VARCHAR(150) NOT NULL,
    price DECIMAL(12,2) NOT NULL,
    stock INT NOT NULL DEFAULT 0,
    minimum_stock INT NOT NULL DEFAULT 5,
    FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE RESTRICT
);

CREATE TABLE stock_movements (
    id INT AUTO_INCREMENT PRIMARY KEY,
    product_id INT NOT NULL,
    user_id INT NOT NULL,
    type ENUM('IN', 'OUT') NOT NULL,
    quantity INT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (product_id) REFERENCES products(id) ON DELETE CASCADE,
    FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE RESTRICT
);

-- Seed Data Awal
INSERT INTO users (username, password_hash, role) VALUES
('admin', '{admin_hash}', 'admin'),
('staff', '{staff_hash}', 'staff');

INSERT INTO categories (id, name) VALUES (1, 'Kalung'), (2, 'Gelang'), (3, 'Cincin');
INSERT INTO suppliers (id, name, phone) VALUES (1, 'Supplier Aksesoris Glow', '08123456789');

INSERT INTO products (id, category_id, sku, name, price, stock, minimum_stock) VALUES
(1, 1, 'KLG-001', 'Kalung Emas Titanium Minimalis', 125000.00, 15, 5),
(2, 2, 'GLG-002', 'Gelang Serut Mutiara Air Tawar', 85000.00, 3, 5),
(3, 3, 'CCN-003', 'Cincin Perak Solitaire Permata', 95000.00, 20, 5);

-- Sample Data Transaksi Awal agar Dashboard tidak kosong
INSERT INTO stock_movements (product_id, user_id, type, quantity) VALUES
(1, 1, 'IN', 15),
(2, 1, 'IN', 5),
(2, 2, 'OUT', 2),
(3, 1, 'IN', 20);
"""

with open("schema.sql", "w") as f:
    f.write(schema_sql)

os.system("mysql -u root < schema.sql")

# 3. Buat File Aplikasi PHP (Tema Krem & Kuning Soft + Edit/Hapus)
print("--- [3/4] Membuat Aplikasi Web PHP (Tema Krem & Kuning Soft) ---")

app_code = """<?php
session_start();

$host = 'localhost';
$db   = 'aksesoris_inventory';
$user = 'root';
$pass = '';

try {
    $pdo = new PDO("mysql:host=$host;dbname=$db;charset=utf8", $user, $pass, [
        PDO::ATTR_ERRMODE => PDO::ERRMODE_EXCEPTION,
        PDO::ATTR_DEFAULT_FETCH_MODE => PDO::FETCH_ASSOC
    ]);
} catch (PDOException $e) {
    die("Koneksi Database Gagal: " . $e->getMessage());
}

function escape($str) {
    return htmlspecialchars($str ?? '', ENT_QUOTES, 'UTF-8');
}

$action = $_GET['action'] ?? 'dashboard';

// --- LOGIN ---
if ($action === 'login') {
    if ($_SERVER['REQUEST_METHOD'] === 'POST') {
        $username = $_POST['username'] ?? '';
        $password = $_POST['password'] ?? '';

        $stmt = $pdo->prepare("SELECT * FROM users WHERE username = :username");
        $stmt->execute(['username' => $username]);
        $u = $stmt->fetch();

        if ($u && password_verify($password, $u['password_hash'])) {
            session_regenerate_id(true);
            $_SESSION['user'] = ['id' => $u['id'], 'username' => $u['username'], 'role' => $u['role']];
            header("Location: index.php?action=dashboard");
            exit;
        } else {
            $login_error = "Username atau password salah!";
        }
    }
    ?>
    <!DOCTYPE html>
    <html>
    <head><title>Login - Inventory Aksesoris</title>
    <style>
    body{font-family:'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background:#FAF6EE; display:flex; justify-content:center; align-items:center; height:100vh; margin:0;}
    .card{background:#FFFDF9; padding:30px; border-radius:12px; border: 1px solid #EFE3CF; box-shadow:0 4px 15px rgba(180,160,130,0.15); width:320px; color:#4A3E3D;}
    h2{margin-top:0; color:#5C4B37; text-align:center;}
    input,button{width:100%; padding:11px; margin:8px 0; box-sizing:border-box; border-radius:6px; border:1px solid #DDD0BD;}
    button{background:#FFE599; color:#5C4100; border:none; font-weight:bold; cursor:pointer; font-size:15px; transition:0.2s;}
    button:hover{background:#F5D061;}
    .info-box{background:#FFF2C2; padding:10px; border-radius:6px; font-size:12px; margin-top:15px; color:#5C4100;}
    </style></head>
    <body>
    <div class="card">
        <h2>Login System</h2>
        <?php if(!empty($login_error)) echo "<p style='color:#C62828; font-weight:bold; text-align:center;'>$login_error</p>"; ?>
        <form method="POST">
            <label><b>Username</b></label>
            <input type="text" name="username" placeholder="Masukkan username" required>
            <label><b>Password</b></label>
            <input type="password" name="password" placeholder="Masukkan password" required>
            <button type="submit">LOGIN</button>
        </form>
        <div class="info-box">
            <b>Default Login:</b><br>
            • Admin: <code>admin</code> / <code>admin123</code><br>
            • Staff: <code>staff</code> / <code>staff123</code>
        </div>
    </div>
    </body>
    </html>
    <?php
    exit;
}

if ($action === 'logout') {
    session_destroy();
    header("Location: index.php?action=login");
    exit;
}

if (!isset($_SESSION['user'])) {
    header("Location: index.php?action=login");
    exit;
}

$user = $_SESSION['user'];

// --- TAMBAH PRODUK BARU ---
if ($action === 'product_add' && $_SERVER['REQUEST_METHOD'] === 'POST') {
    $sku = trim($_POST['sku'] ?? '');
    $name = trim($_POST['name'] ?? '');
    $category_id = (int)($_POST['category_id'] ?? 0);
    $price = (float)($_POST['price'] ?? 0);
    $stock = (int)($_POST['stock'] ?? 0);
    $minimum_stock = (int)($_POST['minimum_stock'] ?? 5);

    if(!empty($sku) && !empty($name) && $category_id > 0 && $price >= 0) {
        try {
            $stmt = $pdo->prepare("INSERT INTO products (sku, name, category_id, price, stock, minimum_stock) VALUES (:sku, :name, :cat, :price, :stock, :min)");
            $stmt->execute(['sku'=>$sku, 'name'=>$name, 'cat'=>$category_id, 'price'=>$price, 'stock'=>$stock, 'min'=>$minimum_stock]);
            $_SESSION['msg_success'] = "Produk berhasil ditambahkan!";
        } catch (Exception $e) {
            $_SESSION['msg_error'] = "Gagal menambah produk: " . $e->getMessage();
        }
    } else {
        $_SESSION['msg_error'] = "Mohon lengkapi seluruh field dengan benar!";
    }
    header("Location: index.php?action=products");
    exit;
}

// --- EDIT PRODUK (PROSES) ---
if ($action === 'product_edit_process' && $_SERVER['REQUEST_METHOD'] === 'POST') {
    $id = (int)($_POST['id'] ?? 0);
    $sku = trim($_POST['sku'] ?? '');
    $name = trim($_POST['name'] ?? '');
    $category_id = (int)($_POST['category_id'] ?? 0);
    $price = (float)($_POST['price'] ?? 0);
    $stock = (int)($_POST['stock'] ?? 0);
    $minimum_stock = (int)($_POST['minimum_stock'] ?? 5);

    if($id > 0 && !empty($sku) && !empty($name) && $category_id > 0) {
        try {
            $stmt = $pdo->prepare("UPDATE products SET sku = :sku, name = :name, category_id = :cat, price = :price, stock = :stock, minimum_stock = :min WHERE id = :id");
            $stmt->execute(['sku'=>$sku, 'name'=>$name, 'cat'=>$category_id, 'price'=>$price, 'stock'=>$stock, 'min'=>$minimum_stock, 'id'=>$id]);
            $_SESSION['msg_success'] = "Produk berhasil diperbarui!";
        } catch (Exception $e) {
            $_SESSION['msg_error'] = "Gagal memperbarui produk: " . $e->getMessage();
        }
    }
    header("Location: index.php?action=products");
    exit;
}

// --- HAPUS PRODUK ---
if ($action === 'product_delete') {
    $id = (int)($_GET['id'] ?? 0);
    if ($id > 0) {
        try {
            $stmt = $pdo->prepare("DELETE FROM products WHERE id = :id");
            $stmt->execute(['id' => $id]);
            $_SESSION['msg_success'] = "Produk berhasil dihapus!";
        } catch (Exception $e) {
            $_SESSION['msg_error'] = "Gagal menghapus produk: " . $e->getMessage();
        }
    }
    header("Location: index.php?action=products");
    exit;
}

// --- TRANSAKSI STOK ---
if ($action === 'transaction' && $_SERVER['REQUEST_METHOD'] === 'POST') {
    $product_id = filter_input(INPUT_POST, 'product_id', FILTER_VALIDATE_INT);
    $type       = $_POST['type'] ?? '';
    $quantity   = filter_input(INPUT_POST, 'quantity', FILTER_VALIDATE_INT);

    if (!$product_id || !$quantity || $quantity <= 0 || !in_array($type, ['IN', 'OUT'])) {
        $_SESSION['msg_error'] = "Input transaksi tidak valid!";
    } else {
        try {
            $pdo->beginTransaction();

            $stmt = $pdo->prepare("SELECT stock FROM products WHERE id = :id FOR UPDATE");
            $stmt->execute(['id' => $product_id]);
            $prod = $stmt->fetch();

            if (!$prod) {
                throw new Exception("Produk tidak ditemukan.");
            }

            if ($type === 'OUT' && $prod['stock'] < $quantity) {
                throw new Exception("Gagal: Stok tidak mencukupi!");
            }

            if ($type === 'IN') {
                $stmtUpd = $pdo->prepare("UPDATE products SET stock = stock + :qty WHERE id = :id");
            } else {
                $stmtUpd = $pdo->prepare("UPDATE products SET stock = stock - :qty WHERE id = :id");
            }
            $stmtUpd->execute(['qty' => $quantity, 'id' => $product_id]);

            $stmtHist = $pdo->prepare("INSERT INTO stock_movements (product_id, user_id, type, quantity) VALUES (:p_id, :u_id, :type, :qty)");
            $stmtHist->execute(['p_id' => $product_id, 'u_id' => $user['id'], 'type' => $type, 'qty' => $quantity]);

            $pdo->commit();
            $_SESSION['msg_success'] = "Transaksi $type berhasil diproses!";
        } catch (Exception $e) {
            $pdo->rollBack();
            $_SESSION['msg_error'] = $e->getMessage();
        }
    }
    header("Location: index.php?action=products");
    exit;
}

if ($action === 'admin_only') {
    if ($user['role'] !== 'admin') {
        http_response_code(403);
        echo "<h2 style='color:#C62828;'>403 Forbidden: Hanya ADMIN yang boleh mengakses halaman ini!</h2><a href='index.php'>Kembali</a>";
        exit;
    }
}
?>
<!DOCTYPE html>
<html>
<head>
    <title>Inventory Aksesoris Online</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; padding: 20px; background: #FAF6EE; color: #4A3E3D; }
        .nav { background: #EFE3CF; padding: 12px 20px; border-radius: 10px; color: #4A3E3D; border: 1px solid #E5D3B3; margin-bottom: 25px; display: flex; justify-content: space-between; align-items: center; box-shadow: 0 2px 5px rgba(0,0,0,0.03); }
        .nav a { color: #5C4B37; margin-left: 15px; text-decoration: none; font-weight: 600; padding: 6px 12px; border-radius: 6px; transition: 0.2s; }
        .nav a:hover { background: #FFE89C; color: #5C4100; }

        .card { background: #FFFDF9; padding: 20px; border-radius: 10px; border: 1px solid #EFE3CF; box-shadow: 0 2px 8px rgba(180,160,130,0.08); margin-bottom: 25px; }

        table { width: 100%; border-collapse: collapse; background: #FFFDF9; border-radius: 8px; overflow: hidden; }
        th, td { border: 1px solid #EFE3CF; padding: 12px; text-align: left; }
        th { background-color: #FFE699; color: #5C4100; font-weight: bold; }
        tr:nth-child(even) { background-color: #FAF6EE; }

        .badge-danger { background: #F8D7DA; color: #721C24; padding: 4px 8px; border-radius: 4px; font-size: 11px; font-weight: bold; border: 1px solid #F5C6CB; }
        .badge-in { background: #D4EDDA; color: #155724; padding: 3px 6px; border-radius: 4px; font-weight: bold; }
        .badge-out { background: #F8D7DA; color: #721C24; padding: 3px 6px; border-radius: 4px; font-weight: bold; }

        .btn { padding: 6px 12px; text-decoration: none; border-radius: 6px; font-weight: bold; font-size: 13px; border: none; cursor: pointer; display: inline-block; transition: 0.2s; }
        .btn-yellow { background: #FFE89C; color: #5C4100; border: 1px solid #F5D061; }
        .btn-yellow:hover { background: #F5D061; }

        .btn-edit { background: #FFF2C2; color: #664D00; border: 1px solid #FFE699; }
        .btn-edit:hover { background: #FFE89C; }

        .btn-delete { background: #FAD4D4; color: #7A1C1C; border: 1px solid #F5B7B7; }
        .btn-delete:hover { background: #F7A8A8; }

        .btn-green { background: #D4EDDA; color: #155724; border: 1px solid #C3E6CB; }
        .btn-red { background: #F8D7DA; color: #721C24; border: 1px solid #F5C6CB; }

        .alert-danger { color: #721C24; background: #F8D7DA; padding: 12px; border-radius: 8px; margin-bottom: 15px; border: 1px solid #F5C6CB; }
        .alert-success { color: #155724; background: #D4EDDA; padding: 12px; border-radius: 8px; margin-bottom: 15px; border: 1px solid #C3E6CB; }

        input, select { padding: 8px 12px; border: 1px solid #DDD0BD; border-radius: 6px; background: #FFFDF9; color: #4A3E3D; }
        input:focus, select:focus { outline: none; border-color: #F5D061; }
    </style>
</head>
<body>

<div class="nav">
    <div>
        <span style="font-size:18px; font-weight:bold; color:#5C4B37;">✨ Aksesoris Store Inventory</span>
        <span style="font-size:13px; color:#8C7A6B; margin-left:10px;">(User: <b><?= escape($user['username']) ?></b> | Role: <b><?= escape($user['role']) ?></b>)</span>
    </div>
    <div>
        <a href="index.php?action=dashboard">Dashboard</a>
        <a href="index.php?action=products">Produk & Transaksi</a>
        <a href="index.php?action=admin_only">Kelola User (Admin)</a>
        <a href="index.php?action=logout" style="background:#FAD4D4; color:#7A1C1C;">Logout</a>
    </div>
</div>

<?php
if (!empty($_SESSION['msg_error'])) {
    echo "<div class='alert-danger'>".escape($_SESSION['msg_error'])."</div>";
    unset($_SESSION['msg_error']);
}
if (!empty($_SESSION['msg_success'])) {
    echo "<div class='alert-success'>".escape($_SESSION['msg_success'])."</div>";
    unset($_SESSION['msg_success']);
}

// === DASHBOARD AGREGAT ===
if ($action === 'dashboard') {
    $total_prod = $pdo->query("SELECT COUNT(id) FROM products")->fetchColumn();
    $total_val  = $pdo->query("SELECT SUM(stock * price) FROM products")->fetchColumn();
    $low_stock  = $pdo->query("SELECT COUNT(id) FROM products WHERE stock <= minimum_stock")->fetchColumn();
    ?>
    <h2 style="color:#5C4B37;">Dashboard Laporan Inventory</h2>
    <div style="display: flex; gap: 20px; margin-bottom: 20px;">
        <div class="card" style="flex: 1;">
            <h3 style="margin-top:0; color:#8C7A6B;">Total Produk</h3>
            <p style="font-size: 28px; font-weight: bold; margin:0; color:#5C4B37;"><?= $total_prod ?></p>
        </div>
        <div class="card" style="flex: 1;">
            <h3 style="margin-top:0; color:#8C7A6B;">Nilai Total Inventory</h3>
            <p style="font-size: 28px; font-weight: bold; margin:0; color:#2E7D32;">Rp <?= number_format($total_val ?? 0, 2) ?></p>
        </div>
        <div class="card" style="flex: 1;">
            <h3 style="margin-top:0; color:#8C7A6B;">Stok Rendah (&le; Min)</h3>
            <p style="font-size: 28px; font-weight: bold; margin:0; color:#C62828;"><?= $low_stock ?></p>
        </div>
    </div>

    <div class="card">
        <h3 style="margin-top:0; color:#5C4B37;">Histori Transaksi Terakhir (Stock Movements)</h3>
        <table>
            <tr><th>Waktu</th><th>SKU</th><th>Nama Produk</th><th>Tipe</th><th>Jumlah</th><th>User</th></tr>
            <?php
            $stmtH = $pdo->query("SELECT sm.*, p.sku, p.name as p_name, u.username FROM stock_movements sm JOIN products p ON p.id=sm.product_id JOIN users u ON u.id=sm.user_id ORDER BY sm.id DESC LIMIT 10");
            $histories = $stmtH->fetchAll();
            if (count($histories) === 0):
            ?>
                <tr><td colspan="6" style="text-align:center; color:#888;">Belum ada histori transaksi.</td></tr>
            <?php else: ?>
                <?php foreach($histories as $h): ?>
                <tr>
                    <td><?= escape($h['created_at']) ?></td>
                    <td><b><?= escape($h['sku']) ?></b></td>
                    <td><?= escape($h['p_name']) ?></td>
                    <td><span class="<?= $h['type']=='IN'?'badge-in':'badge-out' ?>"><?= escape($h['type']) ?></span></td>
                    <td><b><?= escape($h['quantity']) ?></b></td>
                    <td><?= escape($h['username']) ?></td>
                </tr>
                <?php endforeach; ?>
            <?php endif; ?>
        </table>
    </div>
    <?php
}

// === EDIT FORM PAGE ===
if ($action === 'product_edit') {
    $id = (int)($_GET['id'] ?? 0);
    $stmtP = $pdo->prepare("SELECT * FROM products WHERE id = :id");
    $stmtP->execute(['id'=>$id]);
    $edit_prod = $stmtP->fetch();
    $categories = $pdo->query("SELECT * FROM categories")->fetchAll();

    if ($edit_prod):
    ?>
    <div class="card" style="max-width:600px; margin:0 auto;">
        <h2 style="margin-top:0; color:#5C4B37;">Edit Data Produk</h2>
        <form method="POST" action="index.php?action=product_edit_process">
            <input type="hidden" name="id" value="<?= $edit_prod['id'] ?>">
            <div style="margin-bottom:12px;">
                <label><b>SKU:</b></label><br>
                <input type="text" name="sku" value="<?= escape($edit_prod['sku']) ?>" required style="width:100%; box-sizing:border-box;">
            </div>
            <div style="margin-bottom:12px;">
                <label><b>Nama Produk:</b></label><br>
                <input type="text" name="name" value="<?= escape($edit_prod['name']) ?>" required style="width:100%; box-sizing:border-box;">
            </div>
            <div style="margin-bottom:12px;">
                <label><b>Kategori:</b></label><br>
                <select name="category_id" required style="width:100%;">
                    <?php foreach($categories as $cat): ?>
                        <option value="<?= $cat['id'] ?>" <?= $edit_prod['category_id'] == $cat['id'] ? 'selected':'' ?>><?= escape($cat['name']) ?></option>
                    <?php endforeach; ?>
                </select>
            </div>
            <div style="display:flex; gap:10px; margin-bottom:12px;">
                <div style="flex:1;">
                    <label><b>Harga (Rp):</b></label>
                    <input type="number" name="price" step="0.01" value="<?= $edit_prod['price'] ?>" required style="width:100%; box-sizing:border-box;">
                </div>
                <div style="flex:1;">
                    <label><b>Stok Sekarang:</b></label>
                    <input type="number" name="stock" value="<?= $edit_prod['stock'] ?>" required style="width:100%; box-sizing:border-box;">
                </div>
                <div style="flex:1;">
                    <label><b>Min. Stok:</b></label>
                    <input type="number" name="minimum_stock" value="<?= $edit_prod['minimum_stock'] ?>" required style="width:100%; box-sizing:border-box;">
                </div>
            </div>
            <button type="submit" class="btn btn-yellow" style="width:100%; padding:10px;">Simpan Perubahan</button>
            <a href="index.php?action=products" style="display:block; text-align:center; margin-top:10px; color:#777; text-decoration:none;">Batal</a>
        </form>
    </div>
    <?php
    endif;
}

// === LISTING PRODUK & FORM TAMBAH ===
if ($action === 'products') {
    $search = $_GET['search'] ?? '';
    $cat_filter = filter_input(INPUT_GET, 'category_id', FILTER_VALIDATE_INT);

    $sort_allow = ['name' => 'p.name', 'price' => 'p.price', 'stock' => 'p.stock'];
    $sort_by = $_GET['sort'] ?? 'name';
    $order_by = $sort_allow[$sort_by] ?? 'p.name';

    $page = max(1, (int)($_GET['page'] ?? 1));
    $perPage = 5;
    $offset = ($page - 1) * $perPage;

    $where = ["1=1"];
    $params = [];

    if ($search !== '') {
        $where[] = "(p.name LIKE :s OR p.sku LIKE :s)";
        $params['s'] = "%$search%";
    }
    if ($cat_filter) {
        $where[] = "p.category_id = :c";
        $params['c'] = $cat_filter;
    }

    $where_sql = implode(" AND ", $where);

    $stmtCount = $pdo->prepare("SELECT COUNT(*) FROM products p WHERE $where_sql");
    $stmtCount->execute($params);
    $totalData = $stmtCount->fetchColumn();
    $totalPages = ceil($totalData / $perPage);

    $sql = "SELECT p.*, c.name AS category_name FROM products p
            JOIN categories c ON c.id = p.category_id
            WHERE $where_sql
            ORDER BY $order_by ASC
            LIMIT $perPage OFFSET $offset";
    $stmt = $pdo->prepare($sql);
    $stmt->execute($params);
    $products = $stmt->fetchAll();

    $categories = $pdo->query("SELECT * FROM categories")->fetchAll();
    ?>
    <h2 style="color:#5C4B37;">Data Stok Produk & Transaksi</h2>

    <!-- FORM TAMBAH PRODUK BARU -->
    <div class="card">
        <h3 style="margin-top:0; color:#5C4B37;">+ Tambah Produk Aksesoris Baru</h3>
        <form method="POST" action="index.php?action=product_add" style="display:flex; flex-wrap:wrap; gap:10px; align-items:flex-end;">
            <div>
                <label style="font-size:12px; font-weight:bold;">SKU</label><br>
                <input type="text" name="sku" placeholder="misal: KLG-002" required style="width:110px;">
            </div>
            <div>
                <label style="font-size:12px; font-weight:bold;">Nama Produk</label><br>
                <input type="text" name="name" placeholder="Nama Aksesoris" required style="width:180px;">
            </div>
            <div>
                <label style="font-size:12px; font-weight:bold;">Kategori</label><br>
                <select name="category_id" required>
                    <?php foreach($categories as $cat): ?>
                        <option value="<?= $cat['id'] ?>"><?= escape($cat['name']) ?></option>
                    <?php endforeach; ?>
                </select>
            </div>
            <div>
                <label style="font-size:12px; font-weight:bold;">Harga (Rp)</label><br>
                <input type="number" name="price" placeholder="Harga" required style="width:110px;">
            </div>
            <div>
                <label style="font-size:12px; font-weight:bold;">Stok</label><br>
                <input type="number" name="stock" placeholder="Stok" value="10" required style="width:70px;">
            </div>
            <div>
                <label style="font-size:12px; font-weight:bold;">Min. Stok</label><br>
                <input type="number" name="minimum_stock" placeholder="Min" value="5" required style="width:70px;">
            </div>
            <button type="submit" class="btn btn-yellow">+ Tambah</button>
        </form>
    </div>

    <!-- FILTER & PENCARIAN -->
    <div class="card">
        <form method="GET" style="display:flex; gap:10px; flex-wrap:wrap;">
            <input type="hidden" name="action" value="products">
            <input type="text" name="search" placeholder="Cari nama / SKU..." value="<?= escape($search) ?>">
            <select name="category_id">
                <option value="">-- Semua Kategori --</option>
                <?php foreach($categories as $cat): ?>
                    <option value="<?= $cat['id'] ?>" <?= $cat_filter == $cat['id'] ? 'selected':'' ?>><?= escape($cat['name']) ?></option>
                <?php endforeach; ?>
            </select>
            <select name="sort">
                <option value="name" <?= $sort_by=='name'?'selected':'' ?>>Urutkan Nama</option>
                <option value="price" <?= $sort_by=='price'?'selected':'' ?>>Urutkan Harga</option>
                <option value="sort" <?= $sort_by=='stock'?'selected':'' ?>>Urutkan Stok</option>
            </select>
            <button type="submit" class="btn btn-yellow">Filter</button>
        </form>
    </div>

    <!-- TABEL PRODUK -->
    <table>
        <tr>
            <th>SKU</th>
            <th>Nama Produk Aksesoris</th>
            <th>Kategori</th>
            <th>Harga</th>
            <th>Stok</th>
            <th>Aksi Transaksi</th>
            <th>Kelola</th>
        </tr>
        <?php foreach($products as $p): ?>
        <tr>
            <td><b><?= escape($p['sku']) ?></b></td>
            <td><?= escape($p['name']) ?></td>
            <td><?= escape($p['category_name']) ?></td>
            <td>Rp <?= number_format($p['price'], 2) ?></td>
            <td>
                <b><?= $p['stock'] ?></b>
                <?php if($p['stock'] <= $p['minimum_stock']): ?>
                    <br><span class="badge-danger">Stok Rendah</span>
                <?php endif; ?>
            </td>
            <td>
                <form method="POST" action="index.php?action=transaction" style="display:inline-flex; gap:5px;">
                    <input type="hidden" name="product_id" value="<?= $p['id'] ?>">
                    <input type="number" name="quantity" min="1" style="width:55px;" required placeholder="Qty">
                    <button type="submit" name="type" value="IN" class="btn btn-green">+ Masuk</button>
                    <button type="submit" name="type" value="OUT" class="btn btn-red">- Keluar</button>
                </form>
            </td>
            <td>
                <a href="index.php?action=product_edit&id=<?= $p['id'] ?>" class="btn btn-edit">Edit</a>
                <a href="index.php?action=product_delete&id=<?= $p['id'] ?>" class="btn btn-delete" onclick="return confirm('Apakah Anda yakin ingin menghapus produk ini?')">Hapus</a>
            </td>
        </tr>
        <?php endforeach; ?>
    </table>

    <!-- PAGINASI -->
    <div style="margin-top:15px; text-align:center;">
        <b>Halaman: </b>
        <?php for($i=1; $i<=$totalPages; $i++): ?>
            <a href="index.php?action=products&page=<?= $i ?>&search=<?= urlencode($search) ?>&category_id=<?= $cat_filter ?>&sort=<?= $sort_by ?>"
               style="padding:5px 12px; margin:0 2px; border-radius:4px; border:1px solid #FFE699; background:<?= $i==$page?'#F5D061':'#FFFDF9' ?>; color:#5C4100; text-decoration:none; font-weight:bold;">
               <?= $i ?>
            </a>
        <?php endfor; ?>
    </div>
    <?php
}

if ($action === 'admin_only') {
    echo "<div class='card'><h2>Halaman Administrasi Pengguna (Admin Only)</h2><p>Selamat datang di area khusus Admin!</p><a href='index.php' class='btn btn-yellow'>Kembali ke Dashboard</a></div>";
}
?>
</body>
</html>
"""

with open("index.php", "w") as f:
    f.write(app_code)

# 4. Jalankan Server Web PHP & Cloudflare Tunnel
print("--- [4/4] Menjalankan Server Web & Membuat Tautan ---")
subprocess.Popen(["php", "-S", "127.0.0.1:8000"])
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("\nMengambil tautan web (tunggu 5-10 detik)...")
web_url = None
for _ in range(25):
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            web_url = match.group(0)
            break
    time.sleep(0.5)

if web_url:
    print("\n" + "="*60)
    print(" BUKA TAUTAN APLIKASI WEB DI BAWAH INI:")
    print(f" {web_url}")
    print("="*60 + "\n")
else:
    print("\nGagal otomatis membaca URL, silakan periksa output log tunnel.")

--- [1/4] Menginstal PHP, MySQL, & Cloudflare Tunnel ---
--- [2/4] Menyiapkan Database MySQL & Tabel ---
--- [3/4] Membuat Aplikasi Web PHP (Tema Krem & Kuning Soft) ---
--- [4/4] Menjalankan Server Web & Membuat Tautan ---

Mengambil tautan web (tunggu 5-10 detik)...

 BUKA TAUTAN APLIKASI WEB DI BAWAH INI:
 https://duties-boxes-pgp-raw.trycloudflare.com

